In [4]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re


COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
REGEX_ANO = re.compile(r"^(\d{4})")



In [5]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\00.DB\2026.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Desktop\INFORM_27_XLS")
    PASTA_FICHEIROS_OUTPUT = Path(r"C:\Users\LISARR\Documents\python\000.Dados_input\inform_27")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00.DB/2026.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")


✓ BD: C:\Users\LISARR\Documents\python\00.DB\2026.db


In [6]:
import csv

# ==========================
# Converter XLS → CSV (Streaming, sem carregar tudo em RAM)
# ==========================
def ler_xls_streaming(caminho):
    """Retorna (cabecalho, generator de linhas)."""
    tree = ET.parse(caminho)
    raiz = tree.getroot()
    worksheets = raiz.findall(".//ss:Worksheet", NS)

    if not worksheets:
        raise ValueError("Nenhuma worksheet encontrada.")

    def ler_linha(row):
        valores = []
        proximo = 1
        for cell in row.findall("ss:Cell", NS):
            indice = cell.get(SS_INDEX)
            indice = int(indice) if indice else proximo
            while len(valores) < indice - 1:
                valores.append(None)
            data = cell.find("ss:Data", NS)
            valores.append(data.text if data is not None else None)
            proximo = indice + 1
        return valores

    # Extrair cabeçalho da 1ª sheet
    primeira_tabela = worksheets[0].find("ss:Table", NS)
    primeira_row = primeira_tabela.findall("ss:Row", NS)[0]
    cabecalho = ler_linha(primeira_row)
    cabecalho = [str(v).strip() if v else f"COLUNA_{i + 1}" for i, v in enumerate(cabecalho)]

    def gerar_linhas():
        for num_sheet, worksheet in enumerate(worksheets):
            tabela = worksheet.find("ss:Table", NS)
            if tabela is None:
                continue
            rows = tabela.findall("ss:Row", NS)
            if not rows:
                continue
            linhas_comeco = 1 if num_sheet == 0 else 0
            for row in rows[linhas_comeco:]:
                valores = ler_linha(row)
                if len(valores) < len(cabecalho):
                    valores += [None] * (len(cabecalho) - len(valores))
                yield valores[:len(cabecalho)]

    return cabecalho, gerar_linhas()


ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))

if not ficheiros:
    print(f"⚠️  Nenhum ficheiro .xls em: {PASTA_FICHEIROS}")
else:
    total_convertidos = 0

    for numero, caminho in enumerate(ficheiros, 1):
        try:
            contador = 0
            cabecalho, linhas_gen = ler_xls_streaming(caminho)

            with open(PASTA_FICHEIROS_OUTPUT / f"SAL_DAT027.csv", "w", newline="", encoding="utf-8", buffering=8192) as f:
                writer = csv.writer(f)
                writer.writerow(cabecalho)
                for linha in linhas_gen:
                    writer.writerow(linha)
                    contador += 1

            total_convertidos += 1
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} → CSV ({contador:,} linhas)")

        except Exception as erro:
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

    print(f"\n✓ {total_convertidos} ficheiros convertidos")

[1/1] SAL_DAT027 (9)_julho.xls → CSV (10,651 linhas)

✓ 1 ficheiros convertidos
